# Random Shuffle Top-K Evaluation (100 runs)


In [6]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import sys
sys.path.insert(0, r"D:\Politechnika\Deadlift_publikacja_nr_2 — kopia\Skrypty\github")

from utils.datasets import YOLOPoseDataset

In [7]:
N_RANDOM_RUNS = 100
RANDOM_SEED = 42

BASE_ROOT = Path(r"D:\Politechnika\Deadlift_publikacja_nr_2")

TOP_K_BY_SET = {
    "tiger": [10, 25, 50, 100],
    "spinal": [50, 125, 250, 500],
    "standard": [50, 125, 250, 500],
}

VARIANTS = ["axis_out", "angle_preserving_out", "rigid_out"]
DATASETS = ["tiger", "spinal", "standard"]

def build_paths(dataset_name: str, variant: str, top_k: int):
    if dataset_name == "tiger":
        folder = BASE_ROOT / f"tiger_pose_walidacja/tiger-pose_{variant}_{top_k}"
    elif dataset_name == "spinal":
        folder = BASE_ROOT / f"spinal_pose_walidacja/spinal_pose_1000_{variant}_{top_k}"
    elif dataset_name == "standard":
        folder = BASE_ROOT / f"standard_pose_walidacja/pose_standard_flat_{variant}_{top_k}"
    else:
        raise ValueError(f"Nieznany dataset: {dataset_name}")

    train_txt = folder / "train.txt"
    results_txt = folder / "results.txt"
    return train_txt, results_txt

In [8]:
def _normalize_image_path(path_value: str, base_dir: Path) -> str:
    path_value = str(path_value).strip()
    if not path_value:
        return ""

    if os.path.isabs(path_value):
        normalized = os.path.normpath(path_value)
    else:
        normalized = os.path.normpath(str(base_dir / path_value))

    return os.path.normcase(normalized)


def _count_persons_in_label(label_path: Path) -> int:
    with open(label_path, "r", encoding="utf-8") as fh:
        return sum(1 for line in fh if line.strip())


def _build_universe_keys(train_txt_path: Path):
    dataset = YOLOPoseDataset(str(train_txt_path))
    keys = []

    for image_path, label_path in dataset._valid_pairs:
        image_key = _normalize_image_path(image_path, Path(dataset.dataset_dir))
        person_count = _count_persons_in_label(Path(label_path))
        for person_id in range(person_count):
            keys.append((image_key, person_id))

    if not keys:
        raise ValueError(f"Brak probek (image_path, person_id) dla: {train_txt_path}")

    return keys, Path(dataset.dataset_dir)


def _parse_reference_keys(results_txt_path: Path, dataset_dir: Path):
    reference_keys = set()

    with open(results_txt_path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue

            parts = line.split(" | ")
            kv = {
                k.strip(): v.strip()
                for p in parts
                for k, sep, v in [p.partition(": ")]
                if sep
            }

            image_path_raw = kv.get("image_path", "")
            person_id_raw = kv.get("person_id")
            if not image_path_raw or person_id_raw is None:
                continue

            try:
                person_id = int(person_id_raw)
            except (ValueError, TypeError):
                continue

            image_key = _normalize_image_path(image_path_raw, dataset_dir)
            reference_keys.add((image_key, person_id))

    if not reference_keys:
        raise ValueError(f"Nie udalo sie sparsowac kluczy referencyjnych z: {results_txt_path}")

    return reference_keys


def run_single_shuffle(universe_keys, reference_keys, top_k: int):
    ranked = list(universe_keys)
    random.shuffle(ranked)
    top_k = min(int(top_k), len(ranked))
    top_k_set = set(ranked[:top_k])
    return float(len(top_k_set & reference_keys))


def run_randomized_test(dataset_name: str, variant: str, top_k: int, n_runs: int = 100):
    train_txt, results_txt = build_paths(dataset_name, variant, top_k)

    if not train_txt.exists():
        raise FileNotFoundError(f"Brak train.txt: {train_txt}")
    if not results_txt.exists():
        raise FileNotFoundError(f"Brak results.txt: {results_txt}")

    universe_keys, dataset_dir = _build_universe_keys(train_txt)
    reference_keys = _parse_reference_keys(results_txt, dataset_dir)
    universe_set = set(universe_keys)
    reference_in_universe = len(reference_keys & universe_set)

    detected_counts = []
    for _ in range(int(n_runs)):
        detected_counts.append(run_single_shuffle(universe_keys, reference_keys, top_k=top_k))

    arr = np.array(detected_counts, dtype=float)
    return {
        "dataset": dataset_name,
        "variant": variant,
        "top_k": int(top_k),
        "n_runs": int(n_runs),
        "universe_size": int(len(universe_keys)),
        "reference_size": int(len(reference_keys)),
        "reference_in_universe": int(reference_in_universe),
        "mean_detected_top_k": float(np.nanmean(arr)),
        "std_detected_top_k": float(np.nanstd(arr)),
        "min_detected_top_k": float(np.nanmin(arr)),
        "max_detected_top_k": float(np.nanmax(arr)),
    }

In [9]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

rows = []
total_jobs = sum(len(TOP_K_BY_SET[d]) for d in DATASETS) * len(VARIANTS)

with tqdm(total=total_jobs, desc="Konfiguracje") as pbar:
    for variant in VARIANTS:
        for dataset_name in DATASETS:
            for top_k in TOP_K_BY_SET[dataset_name]:
                row = run_randomized_test(
                    dataset_name=dataset_name,
                    variant=variant,
                    top_k=top_k,
                    n_runs=N_RANDOM_RUNS,
                )
                rows.append(row)
                pbar.update(1)

results_df = pd.DataFrame(rows).sort_values(["variant", "dataset", "top_k"]).reset_index(drop=True)
results_df

Konfiguracje:   0%|          | 0/36 [00:00<?, ?it/s]

Konfiguracje: 100%|██████████| 36/36 [05:34<00:00,  9.29s/it]


,dataset,variant,top_k,n_runs,universe_size,reference_size,reference_in_universe,mean_detected_top_k,std_detected_top_k,min_detected_top_k,max_detected_top_k
0,spinal,angle_preserving_out,50,100,1021,50,50,2.55,1.499166,0.0,7.0
1,spinal,angle_preserving_out,125,100,1021,125,125,15.71,3.456284,9.0,24.0
2,spinal,angle_preserving_out,250,100,1021,250,250,61.03,5.221982,46.0,72.0
3,spinal,angle_preserving_out,500,100,1021,500,500,245.86,7.631540,231.0,264.0
4,standard,angle_preserving_out,50,100,990,50,50,2.76,1.631686,0.0,7.0
5,standard,angle_preserving_out,125,100,990,125,125,15.64,3.626348,5.0,23.0
6,standard,angle_preserving_out,250,100,990,250,250,63.12,5.512313,52.0,76.0
7,standard,angle_preserving_out,500,100,990,500,500,252.77,8.421229,232.0,274.0
8,tiger,angle_preserving_out,10,100,263,10,10,0.30,0.500000,0.0,2.0
9,tiger,angle_preserving_out,25,100,263,25,25,2.23,1.263764,0.0,5.0


In [10]:
summary_pivot = results_df.pivot_table(
    index=["variant", "dataset"],
    columns="top_k",
    values="mean_detected_top_k",
    aggfunc="first",
)
summary_pivot

top_k                           10    25    50     100    125    250     500
variant              dataset                                                
angle_preserving_out spinal     NaN   NaN  2.55    NaN  15.71  61.03  245.86
                     standard   NaN   NaN  2.76    NaN  15.64  63.12  252.77
                     tiger     0.30  2.23  9.82  37.49    NaN    NaN     NaN
axis_out             spinal     NaN   NaN  2.38    NaN  15.60  60.87  246.33
                     standard   NaN   NaN  2.69    NaN  15.49  63.90  253.42
                     tiger     0.42  2.28  9.42  38.11    NaN    NaN     NaN
rigid_out            spinal     NaN   NaN  2.55    NaN  15.07  61.73  244.68
                     standard   NaN   NaN  2.50    NaN  16.28  62.89  253.06
                     tiger     0.33  2.58  9.59  37.57    NaN    NaN     NaN